In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType

# 1. Definição do esquema com tipos primitivos e complexos
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("names", ArrayType(StringType()), True),
    StructField("scores", ArrayType(IntegerType()), True),
    StructField("info", StructType([
        StructField("age", IntegerType(), True),
        StructField("city", StringType(), True)
    ]), True),
    StructField("json_str", StringType(), True)
])

# 2. Dados de exemplo
data = [
    (1, ["Alice", "Bob"], [85, 90], {"age": 30, "city": "New York"}, '{"department": "Sales", "level": 3}'),
    (2, ["Carol"], [92, 88], {"age": 25, "city": "San Francisco"}, '{"department": "Engineering", "level": 5}')
]

# 3. Criação do DataFrame
df = spark.createDataFrame(data, schema)

# 4. Registrar View Temporária para consulta via SQL nativo
df.createOrReplaceTempView("temp_complex_data")

display(df)

In [0]:
import pyspark.sql.functions as f

# 1. Tamanho do Array (size)
# 2. Verificar se contém um valor específico (array_contains)
# 3. Dividir texto por caractere separador para gerar Array (split)
df_array_ops = df.select(
    "names",
    f.size("names").alias("names_length"),
    f.array_contains("names", "Alice").alias("contains_alice"),
    f.split("info.city", " ").alias("city_split")
)

display(df_array_ops)

In [0]:
%sql
SELECT 
    names,
    size(names) AS names_length,
    array_contains(names, 'Alice') AS contains_alice,
    split(info.city, ' ') AS city_split
FROM temp_complex_data

In [0]:
from pyspark.sql.functions import explode

# Transforma cada elemento da lista 'names' em uma nova linha no DataFrame
df_exploded = df.select("id", explode("names").alias("name"))

display(df_exploded)

In [0]:
%sql
SELECT 
    id, 
    explode(names) AS name 
FROM temp_complex_data;

In [0]:
# Acessando sub-campos dentro da coluna 'info' usando notação de ponto
df_struct_ops = df.select(
    "id",
    "info.age",
    "info.city"
)

# Alternativa usando withColumn:
# df_struct_ops = df.withColumn("age", f.col("info.age")).withColumn("city", f.col("info.city"))

display(df_struct_ops)

In [0]:
%sql
SELECT 
    id,
    info.age,
    info.city
FROM temp_complex_data;

In [0]:
from pyspark.sql.functions import get_json_object

# Extraindo chaves específicas da string JSON usando caminho Path ($)
df_json_extract = df.select(
    "json_str",
    get_json_object("json_str", "$.department").alias("department"),
    get_json_object("json_str", "$.level").alias("level")
)

display(df_json_extract)

In [0]:
%sql
SELECT 
    from_json(json_str, 'STRUCT<department: STRING, level: INT>').department AS department,
    from_json(json_str, 'STRUCT<department: STRING, level: INT>').level AS level
FROM temp_complex_data;